# Praktikum 11.1?11.3: CNN di Google Colab

Notebook ini menyatukan **Praktikum 11.1 ? Praktikum 11.2 ? Praktikum 11.3** dalam urutan tersebut.

**Runtime:** sebelum menjalankan, pilih **Runtime ? Ubah jenis runtime ? Python 3 ? GPU T4 ? Simpan**. Ketersediaan T4 tetap mengikuti kuota dan ketersediaan Colab.

Jalankan sel setup sekali di awal. Sel tersebut menyiapkan GPU dan mengambil dataset dari repo ini.


In [ ]:
from pathlib import Path
import os
import subprocess
import tensorflow as tf

repo_url = "https://github.com/Shafw00n/Modul-11-KB.git"
repo_dir = Path("/content/Modul-11-KB")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)
os.chdir(repo_dir)

required_paths = ["Dataset 1/Dataset", "Dataset 2/Dataset", "Dataset 3/Dataset"]
missing_paths = [path for path in required_paths if not Path(path).is_dir()]
if missing_paths:
    raise FileNotFoundError(f"Folder dataset tidak ditemukan: {missing_paths}")

physical_gpus = tf.config.list_physical_devices("GPU")
if physical_gpus:
    print(f"GPU aktif: {physical_gpus[0].name}")
else:
    print("GPU tidak terdeteksi. Pilih Runtime ? Ubah jenis runtime ? GPU T4, lalu jalankan ulang.")
print(f"Repo dan dataset siap di: {Path.cwd()}")


## Praktikum 11.1: Klasifikasi Citra Menggunakan CNN (Citra Mobil dan Motor)

**Capaian Pembelajaran:**
1. Memahami arsitektur Convolutional Neural Network (CNN) untuk data citra spasial.
2. Melakukan data loading dan splitting data training & validasi menggunakan `ImageDataGenerator`.
3. Membangun model CNN menggunakan layer Conv2D, MaxPooling2D, Flatten, Dense, dan Dropout.
4. Melatih model dengan optimizer Adam dan Binary Crossentropy.
5. Mengevaluasi performa model dengan kurva Loss/Accuracy, Confusion Matrix, dan Classification Report.

### Langkah 1: Import Library dan Inisialisasi Parameter
Mengimpor library `numpy`, `matplotlib.pyplot`, TensorFlow Keras, dan Scikit-Learn metrics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, classification_report
import os

# PARAMETER
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20
dataset_path = "Dataset 1/Dataset" if os.path.exists("Dataset 1/Dataset") else "Dataset"

print("Parameter selesai diinisialisasi.")

### Langkah 2: Pembagian Dataset Training dan Testing (Validasi)
Dataset dibagi menjadi 70% data training dan 30% data validasi menggunakan `ImageDataGenerator(validation_split=0.3)`. Citra dinormalisasi ke rentang [0, 1] melalui `rescale=1./255`.

In [ ]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.3)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True
)

validation_data = datagen.flow_from_directory(
    dataset_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False
)

total_data = train_data.samples + validation_data.samples
print("====================================")
print("JUMLAH DATA")
print("====================================")
print("Total data :", total_data)
print("Data training :", train_data.samples)
print("Data validasi :", validation_data.samples)
print("Training :", round(train_data.samples/total_data*100, 2), "%")
print("Validasi :", round(validation_data.samples/total_data*100, 2), "%")
print("Class mapping :", train_data.class_indices)

### Langkah 3: Membangun Arsitektur Model CNN
Arsitektur terdiri dari:
- 3 blok Konvolusi + MaxPooling bertingkat (32, 64, 128 filter)
- Flatten layer
- Dense hidden layer 128 unit (ReLU)
- Dropout 0.5 (mencegah overfitting)
- Dense output 1 unit dengan fungsi aktivasi Sigmoid (Klasifikasi Biner Mobil vs Motor)

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

### Langkah 4: Pelatihan Model dan Visualisasi Grafik Akurasi & Loss
Model dilatih selama 20 epoch. Nilai akurasi dan loss pada data training dan validasi divisualisasikan dalam grafik pergerakan per epoch.

In [ ]:
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=EPOCHS
)

# Grafik Accuracy dan Loss
plt.figure(figsize=(10, 8))
plt.subplot(2, 1, 1)
plt.plot(history.history["accuracy"], label="Training", marker='o')
plt.plot(history.history["val_accuracy"], label="Validasi", marker='s')
plt.title("Akurasi Model CNN")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(history.history["loss"], label="Training", marker='o', color='orange')
plt.plot(history.history["val_loss"], label="Validasi", marker='s', color='red')
plt.title("Loss Model CNN")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### Langkah 5: Evaluasi Model, Confusion Matrix, dan Classification Report
Melakukan pengujian pada data validasi secara komprehensif untuk mengukur Precision, Recall, dan F1-Score serta menyimpan model berformat `.keras`.

In [ ]:
loss, accuracy = model.evaluate(validation_data)
print("====================================")
print("HASIL VALIDASI")
print("====================================")
print("Validation Loss     :", round(loss, 4))
print("Validation Accuracy :", round(accuracy * 100, 2), "%")

validation_data.reset()
hasil = model.predict(validation_data)
y_pred = (hasil >= 0.5).astype(int).reshape(-1)
y_true = validation_data.classes

cm = confusion_matrix(y_true, y_pred)
print("\n====================================")
print("CONFUSION MATRIX")
print("====================================")
print(cm)

nama_kelas = list(validation_data.class_indices.keys())
print("\n====================================")
print("CLASSIFICATION REPORT")
print("====================================")
print(classification_report(y_true, y_pred, target_names=nama_kelas))

model.save("model_motor_mobil.keras")
print("[INFO] Model berhasil disimpan ke 'model_motor_mobil.keras'")

## Praktikum 11.2: Membuat Model Klasifikasi 2 Kelas & Pengujian Citra Uji (tes1.jpeg & tes2.jpeg)

**Capaian Pembelajaran:**
1. Memisahkan modul *training* (`train.py`) dan modul *inference/testing* (`test.py`).
2. Melatih model klasifikasi biner citra Mobil dan Motor dengan batch size kecil (2).
3. Menyimpan model terlatih ke dalam format Keras (`model_motor_mobil.keras`).
4. Menguji citra baru (`tes1.jpeg` dan `tes2.jpeg`) menggunakan `load_model()` dan memetakan nilai probabilitas output Sigmoid ke label kelas.

### Bagian 1: Pelatihan Model (Train Module)
Membaca dataset citra mobil dan motor, melatih arsitektur CNN 2 blok konvolusi ber-padding same, dan menyimpan model.

In [ ]:
import tensorflow as tf
import numpy as np
import random
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

dataset_dir = "Dataset 2/Dataset" if os.path.exists("Dataset 2/Dataset") else "Dataset"

datagen = ImageDataGenerator(rescale=1./255)
train_data = datagen.flow_from_directory(
    dataset_dir,
    target_size=(224, 224),
    batch_size=2,
    class_mode="binary",
    shuffle=True,
    seed=42
)

print("Jumlah data :", train_data.samples)
print("Kelas       :", train_data.class_indices)

model = Sequential([
    Input(shape=(224, 224, 3)),
    Conv2D(32, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.fit(train_data, epochs=12)
model.save("model_motor_mobil.keras")
print("[INFO] Model berhasil disimpan ke model_motor_mobil.keras")

### Bagian 2: Pengujian Model Citra Baru (Inference Module)
Memuat model dari disk dan menguji dua citra uji:
- `tes1.jpeg` (Citra Mobil) -> Target output: < 0.5 (MOBIL)
- `tes2.jpeg` (Citra Motor) -> Target output: >= 0.5 (MOTOR)

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

loaded_model = load_model("model_motor_mobil.keras")

def uji_gambar(nama_file):
    if not os.path.exists(nama_file):
        alt_path = os.path.join("Dataset 2", nama_file)
        if os.path.exists(alt_path):
            nama_file = alt_path
    
    img_pil = image.load_img(nama_file, target_size=(224, 224))
    img_arr = image.img_to_array(img_pil) / 255.0
    img_input = np.expand_dims(img_arr, axis=0)

    hasil = loaded_model.predict(img_input, verbose=0)
    nilai = float(hasil[0][0])
    label = "MOTOR" if nilai >= 0.5 else "MOBIL"

    plt.figure(figsize=(4, 4))
    plt.imshow(img_pil)
    plt.title(f"{nama_file}\nPrediksi: {nilai:.6f} -> {label}")
    plt.axis('off')
    plt.show()

    print(f"File: {nama_file}")
    print(f"Nilai prediksi : {nilai}")
    print(f"Hasil          : {label}\n")

uji_gambar("tes1.jpeg")
uji_gambar("tes2.jpeg")

## Praktikum 11.3: Membuat Model Klasifikasi Multi Kelas (5 Kelas Makanan Tradisional & Sayuran)

**Capaian Pembelajaran:**
1. Memahami perbedaan klasifikasi multi-kelas (*multiclass classification*) dibanding klasifikasi biner.
2. Menggunakan fungsi aktivasi Softmax pada layer output untuk menghasilkan distribusi probabilitas terstandarisasi.
3. Menggunakan fungsi loss *Categorical Crossentropy*.
4. Menguji citra baru (`tes0.jpeg` hingga `tes4.jpeg`) dan mengambil probabilitas argmax untuk menentukan kelas prediksi.

### Bagian 1: Pelatihan Model CNN 5 Kelas
Membaca dataset citra 5 kelas makanan (Bakwan Sayur, Bubur Tinotuan, Nasi Beras Merah, Plencing Kangkung, Wortel), melatih model, dan menyimpannya ke `model_makanan.keras`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

dataset_makanan = "Dataset 3/Dataset" if os.path.exists("Dataset 3/Dataset") else "Dataset"

datagen = ImageDataGenerator(rescale=1./255)
train_data = datagen.flow_from_directory(
    dataset_makanan,
    target_size=(224, 224),
    batch_size=2,
    class_mode="categorical"
)

print("Jumlah data :", train_data.samples)
print("Kelas       :", train_data.class_indices)

model = Sequential([
    Input(shape=(224, 224, 3)),
    Conv2D(32, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(5, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(train_data, epochs=10)
model.save("model_makanan.keras")
print("[INFO] Model berhasil disimpan ke 'model_makanan.keras'")

### Bagian 2: Pengujian Citra Uji (tes0.jpeg s/d tes4.jpeg)
Menguji ke-5 citra uji satu per satu untuk mengevaluasi prediksi kelas dan probabilitas keyakinan model.

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import matplotlib.pyplot as plt

loaded_model = load_model("model_makanan.keras")

kelas = {
    0: "Bakwan Sayur",
    1: "Bubur Tinotuan",
    2: "Nasi Beras Merah",
    3: "Plencing Kangkung",
    4: "Wortel"
}

plt.figure(figsize=(15, 6))
for i in range(5):
    fname = f"tes{i}.jpeg"
    if not os.path.exists(fname):
        alt = os.path.join("Dataset 3", fname)
        if os.path.exists(alt):
            fname = alt

    img_pil = image.load_img(fname, target_size=(224, 224))
    img_arr = image.img_to_array(img_pil) / 255.0
    img_batch = np.expand_dims(img_arr, axis=0)

    hasil = loaded_model.predict(img_batch, verbose=0)
    index = int(np.argmax(hasil[0]))
    nilai = float(hasil[0][index])

    plt.subplot(1, 5, i+1)
    plt.imshow(img_pil)
    plt.title(f"tes{i}.jpeg\n{kelas[index]}\n({nilai*100:.1f}%)", fontsize=10)
    plt.axis('off')

    print(f"--- Uji {fname} ---")
    print("Nilai probabilitas :", nilai)
    print("Index kelas        :", index)
    print("Kelas terprediksi  :", kelas[index])
    print()

plt.tight_layout()
plt.show()